# 03 — Preprocessing (Feature Selection, Encoding, & Train-Test Split)

Lanjutan dari `02_cleaning.ipynb`, memakai `data/processed/cleaned_data.csv`.

Langkah di notebook ini (sesuai metodologi pada `AGENTS.md`):

1. Memilih fitur mentah sebagai variabel independen (X): `Gender`, `Age (Month)`, `Weight`, `Height`.
2. **Membuang kolom turunan** yang berpotensi data leakage.
3. **Label encoding**: `Gender` (M → 0, F → 1) dan target `Height for Age` (Not Stunted → 0, Stunted → 1).
4. **Train-test split** 80:20 stratified (random_state=42).
5. Menyimpan X, y serta hasil split ke `data/processed/`.

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from joblib import dump

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.20

In [2]:
PROCESSED_PATH = "../data/processed/cleaned_data.csv"

df = pd.read_csv(PROCESSED_PATH)
print("Shape:", df.shape)
print("Kolom:", list(df.columns))

Shape: (40064, 11)
Kolom: ['No.', 'Gender', 'Age (Month)', 'Weight', 'Height', 'Weight for Age', 'Z-Score  W/A', 'Height for Age', 'Z-Score H/A', 'Weight for Height', 'Z-Score W/H']


## 1. Feature Selection & Data Leakage

Dataset memiliki kolom turunan berikut: `Weight for Age`, `Z-Score W/A`,
`Z-Score H/A`, `Weight for Height`, `Z-Score W/H`.

Kolom-kolom ini **WAJIB dibuang** karena merupakan hasil perhitungan yang
**menurunkan label target** (`Height for Age`). Contoh : target
`Height for Age` adalah hasil klasifikasi dari `Z-Score H/A` (anak dikategorikan
*Stunted* bila H/A Z-score < -2). Bila kolom turunan ikut menjadi fitur, model
hanya akan "mencontek" jawaban dari rumus yang sudah ada, sehingga akurasi
menjadi semu dan tidak mencerminkan kemampuan prediksi dari data fisik mentah.

Oleh karena itu variabel independen yang dipakai hanyalah 4 fitur mentah:
`Gender`, `Age (Month)`, `Weight`, `Height`.

In [3]:
TARGET_COL = "Height for Age"
FEATURES = ["Gender", "Age (Month)", "Weight", "Height"]
DERIVED_COLS = [
    "Weight for Age",
    "Z-Score  W/A",
    "Z-Score H/A",
    "Weight for Height",
    "Z-Score W/H",
]

cols_to_drop = DERIVED_COLS + ["No.", TARGET_COL]
df_noderived = df.drop(columns=cols_to_drop)

print("Kolom setelah buang turunan + No. + target:", list(df_noderived.columns))
assert list(df_noderived.columns) == FEATURES, "Fitur X tidak sesuai FEATURES yang diharapkan!"

Kolom setelah buang turunan + No. + target: ['Gender', 'Age (Month)', 'Weight', 'Height']


## 2. Label Encoding

- **Gender**: `M` → 0, `F` → 1 (mengikuti Sempro: 0 = Laki-laki, 1 = Perempuan).
- **Target `Height for Age`**: `Not Stunted` → 0, `Stunted` → 1.

In [4]:
X = df[FEATURES].copy()
y = df[TARGET_COL].map({"Not Stunted": 0, "Stunted": 1})

X["Gender"] = X["Gender"].map({"M": 0, "F": 1})

print("Distribusi Gender setelah encoding:")
print(X["Gender"].value_counts().sort_index())
print()
print("Distribusi target setelah encoding:")
print(y.value_counts().sort_index())

Distribusi Gender setelah encoding:
Gender
0    21985
1    18079
Name: count, dtype: int64

Distribusi target setelah encoding:
Height for Age
0    23864
1    16200
Name: count, dtype: int64


## 3. Train-Test Split (80:20, stratified)

Data dibagi 80% training / 20% testing secara **stratified** menurut target `y`
agar proporsi kelas (Stunted : Not Stunted) terjaga di kedua bagian, dengan
`random_state=42` agar hasil reproducible.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test  shape:", y_test.shape)

X_train shape: (32051, 4)
X_test  shape: (8013, 4)
y_train shape: (32051,)
y_test  shape: (8013,)


## 4. Distribusi Kelas di y_train & y_test

Memastikan proporsi kelas (Stunted ≈ 40%, Not Stunted ≈ 60%) tetap terjaga
pada training set maupun test set.

In [6]:
def show_dist(name, target):
    counts = target.value_counts().sort_index()
    pct = (target.value_counts(normalize=True).sort_index() * 100).round(2)
    print(f"{name}:")
    for cls in [0, 1]:
        label = "Not Stunted" if cls == 0 else "Stunted"
        print(f"    {label} ({cls}): {counts.get(cls, 0):,}  ({pct.get(cls, 0):.2f}%)")
    print()


show_dist("y (full)", y)
show_dist("y_train", y_train)
show_dist("y_test", y_test)

y (full):
    Not Stunted (0): 23,864  (59.56%)
    Stunted (1): 16,200  (40.44%)

y_train:
    Not Stunted (0): 19,091  (59.56%)
    Stunted (1): 12,960  (40.44%)

y_test:
    Not Stunted (0): 4,773  (59.57%)
    Stunted (1): 3,240  (40.43%)



## 5. Simpan Hasil

X, y, serta hasil split disimpan ke `data/processed/` (format `.joblib` untuk
menjaga tipe data).

In [7]:
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

X.to_csv(os.path.join(OUT_DIR, "X_preprocessed.csv"), index=False)
y.to_csv(os.path.join(OUT_DIR, "y_preprocessed.csv"), index=False)

dump(X_train, os.path.join(OUT_DIR, "X_train.joblib"))
dump(X_test, os.path.join(OUT_DIR, "X_test.joblib"))
dump(y_train, os.path.join(OUT_DIR, "y_train.joblib"))
dump(y_test, os.path.join(OUT_DIR, "y_test.joblib"))

print("Tersimpan:")
for f in ["X_preprocessed.csv", "y_preprocessed.csv",
          "X_train.joblib", "X_test.joblib", "y_train.joblib", "y_test.joblib"]:
    print("  -", os.path.join(OUT_DIR, f))

Tersimpan:
  - ../data/processed\X_preprocessed.csv
  - ../data/processed\y_preprocessed.csv
  - ../data/processed\X_train.joblib
  - ../data/processed\X_test.joblib
  - ../data/processed\y_train.joblib
  - ../data/processed\y_test.joblib


## ⚠️ PERHATIAN PENTING — Titik Kunci Anti-Data-Leakage

> **Mulai titik ini, `X_test` dan `y_test` TIDAK BOLEH disentuh oleh proses
> resampling apa pun** (SMOTE, ENN, SMOTE-ENN, oversampling/undersampling
> lain), dan TIDAK BOLEH dipakai untuk fitting scaler/imputer, feature selection,
> maupun hyperparameter tuning.

- **SMOTE-ENN HANYA** diterapkan pada `X_train`/`y_train` di notebook pemodelan.
- `X_test`/`y_test` dibiarkan murni (touchable hanya saat `.predict()` dan evaluasi)
  agar metrik mencerminkan performa pada data yang belum pernah dilihat model,
  bukan pada data yang ikut dipakai saat resampling/tuning.

## Ringkasan

- X: 4 fitur mentah (`Gender` (0/1), `Age (Month)`, `Weight`, `Height`), shape 40.064 × 4.
- y: `Stunted`=1, `Not Stunted`=0.
- Split 80:20 stratified (random_state=42) → `X_train` (32.051), `X_test` (8.013).
- Kolom turunan ber-leakage dibuang. SMOTE-ENN diterapkan hanya pada training set di notebook berikutnya.